## SUFC - Data Scientist Technical Assessment

February 17, 2026 | Chloe Paventi (chloespaventi@gmail.com)

In [18]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
sns.set_style("whitegrid")
sns.set_palette("deep")

ticket_sales = pd.read_csv("../data/ticket_sales.csv") # import ticket sales data
customers = pd.read_csv("../data/customers.csv") # import customers data

### Section 1 - Data Quality Assessment
This section outlines an initial exploratory data analysis of Sheffield United's ticketing and customer datasets, identifying data quality issues and establishing a cleaning and monitoring framework.

In [2]:
# check data shape (number of rows and columns)
print("Ticket Sales Shape:", ticket_sales.shape)
print("Customers Shape:", customers.shape)

Ticket Sales Shape: (43289, 10)
Customers Shape: (277, 9)


In [3]:
ticket_sales.info() # ticket sales: check data types / number of entries

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43289 entries, 0 to 43288
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   transaction_id       43289 non-null  object
 1   purchaser_id         43289 non-null  object
 2   owner_id             43289 non-null  object
 3   transaction_date     43289 non-null  object
 4   client_type          43289 non-null  object
 5   product_description  43289 non-null  object
 6   event_date           43289 non-null  object
 7   area                 43289 non-null  object
 8   row_number           43289 non-null  object
 9   seat_number          43289 non-null  object
dtypes: object(10)
memory usage: 3.3+ MB


In [4]:
customers.info() # customers: check data types / number of entries

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 277 entries, 0 to 276
Data columns (total 9 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   customer_id                  277 non-null    object 
 1   customer_type                277 non-null    object 
 2   country                      263 non-null    object 
 3   district                     249 non-null    object 
 4   ward                         249 non-null    object 
 5   distance_to_bramall_lane_km  249 non-null    float64
 6   age                          168 non-null    float64
 7   gender                       277 non-null    object 
 8   exclude_from_marketing       277 non-null    bool   
dtypes: bool(1), float64(2), object(6)
memory usage: 17.7+ KB


**Initial Observations:**
- The `ticket_sales` dataset contains 43,289 records and 10 columns. All fields are populated, with no missing values observed. This suggests strong completeness in the transactional data.
- The `customers` dataset contains 277 records and 9 columns. While most structural fields are populated, several demographic attributes contain missing values. The following columns have null entries: `country`, `district`, `ward`, `distance_to_bramall_lane_km`, `age`
- Age has the highest proportion of missing values, which may limit the reliability of age-based segmentation. Similarly, incomplete geographic information may affect location-based analysis.
- Overall, the transactional data appears structurally complete, while the customer dataset requires improvement for a proper demographic analysis.


In [5]:
customers["gender"].value_counts(normalize=True) * 100  # customers: check gender distribution

gender
Male           52.346570
Unspecified    40.794224
Female          6.859206
Name: proportion, dtype: float64

In [6]:
customers["gender"] = customers["gender"].replace("Unspecified", np.nan) # customers: replace "Unspecified" in the gender column with null values

Although the `gender` field contained no null values, a subset of records were labelled as "Unspecified." These entries were treated as missing (NaN) for analytical purposes to prevent them from being misinterpreted as a valid demographic segment during analysis.

In [7]:
ticket_sales.duplicated().sum(),customers.duplicated().sum() # check for duplicates in both datasets

(np.int64(0), np.int64(0))

In [8]:
# check that join between ticket_sales and customers on owner_id and customer_id does not result in any missing values
ticket_sales.merge(customers, left_on="owner_id", right_on="customer_id", how="left")["customer_id"].isna().sum()

np.int64(0)

In [9]:
# check that join between ticket_sales and customers on purchaser_id and customer_id does not result in any missing values
ticket_sales.merge(customers, left_on="purchaser_id", right_on="customer_id", how="left")["customer_id"].isna().sum()

np.int64(0)

In [10]:
# convert date columns from object to datetime format for date-based analysis
ticket_sales["transaction_date"] = pd.to_datetime(ticket_sales["transaction_date"])
ticket_sales["event_date"] = pd.to_datetime(ticket_sales["event_date"])

The `transaction_date` and `event_date` columns were converted from text (object) format to datetime format to enable accurate time-based analysis

**Cleaning Steps to Improve Data Quality**

Based on the initial exploration, the following steps are recommended to improve data quality:

- **Handling "unspecified" values**: Some customers were labelled as "unspecified" rather than having a clearly defined gender. These values should be treated as missing to avoid counting them as a separate demographic group in segmentation analysis.

- **Standardising text fields**: Text columns should be converted to lowercase and trimmed to remove extra spaces. This helps prevent duplicate categories caused by inconsistent formatting.

- **Converting date columns**: Date columns (such as `transaction_date` and `event_date`) should be converted from object (text) format to datetime format so that time-based analysis and trend evaluation can be performed accurately.

- **Checking numeric ranges**: Numeric fields such as age and distance should be reviewed to ensure they fall within reasonable ranges. Any unrealistic values should be flagged for further review or excluded from analysis where appropriate.

**Continuous Data Quality Monitoring**

To maintain high data quality standards, the following monitoring framework is recommended:

- **Missing data tracking**: Regularly monitor the proportion of missing values in key demographic fields such as age, district, and distance. Set up automated scripts to alert the team if null values in these columns exceed a certain threshold.

- **Text reformatting**: Apply a reusable text-cleaning function to automatically strip whitespace and standardise casing across object-type columns. This reduces the risk of artificial category duplication caused by inconsistent data entry.

- **Data Entry Controls**: Introduce validation rules at the point of data entry, such as using dropdown selections instead of free-text fields and enforcing logical constraints (e.g., age must fall within a realistic range, dates must be valid and recent). Preventing errors at the source reduces downstream cleaning requirements.

### Section 2 - Key Demographics
Before analysing demographics, it is important to note that the dataset contains 43,289 hospitality seat records but only 277 unique customers. This indicates a highly concentrated customer base with significant repeat purchasing behaviour. Hospitality demand is therefore driven not just by the number of customers, but by the purchasing intensity of a relatively small group of high-frequency buyers. The following analysis focuses on understanding who these customers are and how their profile has evolved over time.

In [11]:
hospitality_attendance = ticket_sales.merge( # perform left join to combine ticket_sales with customers on owner_id and customer_id to get customer details for each ticket sale
    customers,
    left_on="owner_id",
    right_on="customer_id",
    how="left"
)

The `ticket_sales` dataset was merged with the `customers` dataset to attach demographic information to each seat record, allowing target audience analysis to reflect actual hospitality attendance volume rather than just the number of registered customers. High-frequency customers will appear multiple times based on the number of seats purchased.

In [12]:
hospitality_attendance["distance_band"] = pd.cut(
    hospitality_attendance["distance_to_bramall_lane_km"],
    bins=[0, 10, 25, 50, 100, float("inf")], # create distance bands
    labels=["0-10 km", "10-25 km", "25-50 km", "50-100 km", ">100 km"], 
)
distance_distribution = (
    hospitality_attendance["distance_band"]
    .value_counts(normalize=True, sort=False) * 100 # calculate percentage distribution of distance bands among hospitality attendees
)
distance_distribution.round(2).astype(str) + "%" # format percentages to 2 decimal places and add % sign

distance_band
0-10 km      59.45%
10-25 km     35.59%
25-50 km      0.29%
50-100 km     2.51%
>100 km       2.16%
Name: proportion, dtype: object

The majority of hospitality attendance originates within 25 km of Bramall Lane, indicating that demand is primarily local and regional. A smaller proportion of attendees travel over 50 km, suggesting that while the core audience is locally concentrated, there is some broader geographic reach. This supports the view that hospitality engagement is strongly embedded within the surrounding Sheffield business and supporter community.

In [13]:
hospitality_attendance["event_year"] = hospitality_attendance["event_date"].dt.year # extract year from event_date for time-based analysis

In [14]:
customer_type_by_year = pd.crosstab( # calculate percentage distribution of customer types for each year
    hospitality_attendance["event_year"],
    hospitality_attendance["customer_type"],
    normalize="index"
) * 100 

customer_type_by_year.loc["Overall"] = ( # add overall row to show distribution of customer types across all years
    hospitality_attendance["customer_type"]
    .value_counts(normalize=True) * 100 
)

attendance_total = hospitality_attendance.groupby("event_year").size() # calculate total attendance for each year
attendance_total.loc["Overall"] = len(hospitality_attendance) # create attendance total for all years
attendance_table = customer_type_by_year.copy()
attendance_table.insert(0, "Total Attendance", attendance_total) # combine overall attendance total with percentages per year into a single table
attendance_table

customer_type,Total Attendance,Account,Customer
event_year,,,
2024,12201,79.870502,20.129498
2025,20954,78.557793,21.442207
2026,10134,76.534439,23.465561
Overall,43289,78.454111,21.545889


Hospitality attendance is strongly driven by corporate accounts. Overall, 78.45% of total seat attendance is associated with “Account” customers, compared to 21.55% from individual “Customer” buyers.

Although total attendance increased significantly from 12,201 seats in 2024 to 20,954 in 2025, the proportion of corporate accounts remained consistently high. There is a slight downward trend in corporate share, from 79.87% in 2024 to 76.53% in 2026, suggesting modest growth in individual participation over time.

In [15]:
client_type_by_year = pd.crosstab( # calculate percentage of season vs match-by-match attendance for each year
    hospitality_attendance["event_year"],
    hospitality_attendance["client_type"],
    normalize="index"
) * 100

client_type_by_year.loc["Overall"] = ( # add overall row to show client type across all years
    hospitality_attendance["client_type"]
    .value_counts(normalize=True) * 100
)

client_type_by_year.round(2).astype(str) + "%" # format percentages to 2 decimal places and add % sign

client_type,Match by Match Ticket Holder,Season Ticket Holder
event_year,,
2024,29.26%,70.74%
2025,24.61%,75.39%
2026,17.23%,82.77%
Overall,24.19%,75.81%


Season Ticket Holders account for the majority of hospitality seats across both seasons. Year-over-year data shows an increase in reliance on season ticket holders. The proportion of season-based attendance rose from 70.74% in 2024 to 82.77% in 2026, while match-by-match participation declined from 29.26% to 17.23% over the same period.

This highlights the importance of long-term hospitality packages in driving attendance and suggests that retention strategies for season-based clients are commercially critical.

In [16]:
gender_by_year = pd.crosstab( # calculate gender distribution for each year
    hospitality_attendance["event_year"],
    hospitality_attendance["gender"],
    normalize="index"
) * 100

gender_by_year.loc["Overall"] = ( # add overall row to show gender distribution across all years
    hospitality_attendance["gender"]
    .value_counts(normalize=True) * 100
)

gender_by_year.round(2).astype(str) + "%" # format percentages to 2 decimal places and add % sign

gender,Female,Male
event_year,,
2024,24.47%,75.53%
2025,15.58%,84.42%
2026,14.62%,85.38%
Overall,17.74%,82.26%


Hospitality attendance is predominantly male across all seasons. In 2024, males accounted for 75.53% of seat attendance, increasing to 84.42% in 2025 and 85.38% in 2026. This indicates a shift toward a more male-dominated hospitality audience over time. The declining proportion of female attendees suggests that recent hospitality growth has been driven primarily by male customers or male-led corporate bookings.

It is important to note that a proportion of gender data was originally recorded as “Unspecified” and treated as missing for analytical purposes. While the majority of records contain valid gender information, the presence of missing values slightly limits the precision.

In [17]:
hospitality_attendance["age_band"] = pd.cut( # create age bands 
    hospitality_attendance["age"],
    bins=[18, 25, 35, 45, 55, 65, float("inf")], 
    labels=["18-25", "26-35", "36-45", "46-55", "56-65", "65+"],
    right=True
)

age_by_year = pd.crosstab( # calculate age band distribution for each year
    hospitality_attendance["event_year"],
    hospitality_attendance["age_band"],
    normalize="index"
) * 100

age_by_year.loc["Overall"] = ( # add overall row to show age across all years
    hospitality_attendance["age_band"]
    .value_counts(normalize=True) * 100
)

age_by_year.round(2).astype(str) + "%" # format percentages to 2 decimal places and add % sign

age_band,18-25,26-35,36-45,46-55,56-65,65+
event_year,,,,,,
2024,0.94%,2.73%,9.83%,11.8%,43.39%,31.31%
2025,1.08%,3.05%,8.63%,14.72%,39.52%,32.99%
2026,1.12%,3.64%,14.41%,15.34%,33.57%,31.93%
Overall,1.06%,3.12%,10.46%,14.12%,38.97%,32.27%


Hospitality attendance is heavily concentrated within older age groups. Across all seasons, the largest proportion of seats is held by customers aged 56-65 (38.97%), followed by 65+ (32.27%). Combined, customers aged 56 and above account for over 70% of total hospitality attendance. 

Year-over-year trends show a gradual shift within the 36-55 age range. Overall, the data suggests that hospitality demand is primarily driven by older individuals, with emerging growth among the 36-55 segment.

**Target Audience**: 

Based on attendance patterns, the primary target audience for Sheffield United’s hospitality suites consists of locally based corporate customers with strong repeat purchasing behaviour. Corporate “Account” buyers contribute 78.45% of total seat attendance, and Season Ticket Holders account for 75.81% overall, with season-based engagement increasing year-over-year. This indicates that hospitality demand is driven predominantly by long-term, committed customers rather than occasional match-by-match buyers.

Demographically, the audience is heavily male (82.26% overall) and skewed toward older age groups, with over 70% of attendance coming from individuals aged 56 and above. Geographically, the segment is highly concentrated, with more than 95% of attendees located within 25 km of Bramall Lane.

Overall, the core hospitality audience can be characterised as a locally concentrated, male-dominated, older corporate segment with strong season-based loyalty and repeat engagement patterns.


**Machine Learning Techniques**

- **Lookalike Modelling**: By analysing the characteristics of our existing hospitality customers (e.g., age, location, purchase frequency), we can build a profile of a typical hospitality buyer. This model can then be applied to the wider general admission database to identify supporters who have not yet purchased hospitality but share similar traits, enabling targeted campaigns.

- **Demand Forecasting**: Time-series models can be used to predict hospitality demand based on fixture type, opponents, and time of season. This would support more effective capacity planning, pricing decisions, and promotional timing.

- **Recommendation Systems**: Behaviour-based recommendation models can suggest specific hospitality areas or package upgrades based on the purchasing patterns of similar customers.

### Section 3 - Recommendations
This section outlines key strategic insights worth considering to improve hospitality sales.

**Season Ticket Dependency**: Hospitality attendance is increasingly concentrated among Season Ticket Holders, with match-by-match participation declining year over year. While this provides revenue stability and predictable renewals, it may also indicate limited audience expansion. If the majority of seats are occupied by the same customers each season, there is reduced opportunity for new supporters to experience hospitality and enter the conversion funnel. The club should assess whether hospitality is becoming less accessible to first-time buyers and consider introducing limited-entry or trial packages that encourage new customers to sample the experience

**Aging Audience**: Over 70% of hospitality attendance comes from individuals aged 56 and above. While this demographic may currently represent strong purchasing power, it presents a long-term risk. As this audience ages, attendance frequency may decline, and without younger pipeline replacement, overall demand could decrease. A focus on attracting mid-career professionals (36-55) and younger corporate groups could help ensure continuity.

**Increasing Male Audience**: The growing male dominance within hospitality attendance suggests potential untapped opportunity among female supporters and corporate groups. It is worth evaluating whether current marketing materials, messaging, and hospitality environments are unintentionally aligned with traditional male corporate audiences.

**Geographic Saturation**: With over 95% of hospitality attendance coming from within 25 km of Bramall Lane, demand is highly localised. If attendance from 50 km+ remains minimal, this may reflect transport barriers, limited regional awareness, or perceived lack of relevance. The club should working on extending geographic reach beyond the immediate Sheffield area.

**Individual Attendance**: Corporate accounts make the majority of hospitality attendance, while individual customers remain underrepresented. This may suggest that pricing structures and positioning primarily favour business entertainment over leisure consumption. Reducing the perception that hospitality is exclusively a corporate product may create more demand.

**Demographic Data Quality**: Incomplete demographic data, particularly in age and geographic fields, limits the club’s ability to personalise offers and strategically diversify its audience. Without accurate customer insight, segmentation and targeted marketing are constrained. Improving demographic data collection at the point of sale would support more precise audience profiling, enhanced retention strategies, and more informed long-term commercial planning. 